In [2]:
!pip install datasets pandas numpy

In [3]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

In [5]:
import urllib.request

base_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"

files = {
    "train": "en_ewt-ud-train.conllu",
    "dev": "en_ewt-ud-dev.conllu",
    "test": "en_ewt-ud-test.conllu"
}

for split, filename in files.items():
    url = base_url + filename
    urllib.request.urlretrieve(filename=url.split("/")[-1], url=url)
    print(f"{split} dataset downloaded successfully.")

train dataset downloaded successfully.
dev dataset downloaded successfully.
test dataset downloaded successfully.


In [6]:
!pip install conllu

In [7]:
import conllu
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

In [8]:
with open(
    "en_ewt-ud-train.conllu",
    "r",
    encoding="utf-8"
) as file:
    
    train_data = conllu.parse(file.read())

print("Training sentences:", len(train_data))

Training sentences: 12544


In [9]:
with open(
    "en_ewt-ud-dev.conllu",
    "r",
    encoding="utf-8"
) as file:
    
    dev_data = conllu.parse(file.read())


with open(
    "en_ewt-ud-test.conllu",
    "r",
    encoding="utf-8"
) as file:
    
    test_data = conllu.parse(file.read())


print("Training sentences:", len(train_data))
print("Validation sentences:", len(dev_data))
print("Test sentences:", len(test_data))

Training sentences: 12544
Validation sentences: 2001
Test sentences: 2077


In [10]:
print(train_data[0].metadata)

print("\nWords and POS tags:\n")

for token in train_data[0]:
    print(
        f"{token['form']:15} → {token['upos']}"
    )

{'newdoc id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000', 'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001', 'newpar id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-p0001', 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.'}

Words and POS tags:

Al              → PROPN
-               → PUNCT
Zaman           → PROPN
:               → PUNCT
American        → ADJ
forces          → NOUN
killed          → VERB
Shaikh          → PROPN
Abdullah        → PROPN
al              → PROPN
-               → PUNCT
Ani             → PROPN
,               → PUNCT
the             → DET
preacher        → NOUN
at              → ADP
the             → DET
mosque          → NOUN
in              → ADP
the             → DET
town            → NOUN
of              → ADP
Qaim            → PROPN
,               → PUNCT
near            → ADP
the     

In [11]:
train_sentences = []

for sentence in train_data:
    
    tagged_sentence = []
    
    for token in sentence:
        
        word = token["form"]
        tag = token["upos"]
        
        tagged_sentence.append(
            (word, tag)
        )
    
    train_sentences.append(tagged_sentence)

print(
    "Training sentences:",
    len(train_sentences)
)

Training sentences: 12544


In [12]:
tag_set = set()

for sentence in train_sentences:
    
    for word, tag in sentence:
        tag_set.add(tag)

tag_set = sorted(tag_set)

print("Number of POS tags:", len(tag_set))
print("\nPOS Tags:")
print(tag_set)

Number of POS tags: 18

POS Tags:
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']


In [13]:
transition_counts = defaultdict(Counter)

for sentence in train_sentences:

    previous_tag = "<START>"

    for word, tag in sentence:

        transition_counts[previous_tag][tag] += 1

        previous_tag = tag

    # Transition from final tag to END
    transition_counts[previous_tag]["<END>"] += 1

print("Transition counts created successfully.")

Transition counts created successfully.


In [14]:
print("Transitions from DET:")

for tag, count in transition_counts["DET"].most_common(10):
    print(f"DET → {tag}: {count}")

Transitions from DET:
DET → NOUN: 9577
DET → ADJ: 3813
DET → PROPN: 1235
DET → VERB: 309
DET → ADV: 257
DET → NUM: 247
DET → PUNCT: 209
DET → ADP: 187
DET → DET: 161
DET → _: 133


In [15]:
transition_probabilities = {}

all_tags = tag_set + ["<END>"]

for previous_tag, next_tags in transition_counts.items():

    total = sum(next_tags.values())

    transition_probabilities[previous_tag] = {}

    for next_tag in all_tags:

        count = next_tags.get(next_tag, 0)

        probability = (
            (count + 1) /
            (total + len(all_tags))
        )

        transition_probabilities[
            previous_tag
        ][next_tag] = probability

print("Transition probabilities calculated.")

Transition probabilities calculated.


In [16]:
probability = transition_probabilities[
    "DET"
].get("NOUN", 0)

print(
    f"P(NOUN | DET) = {probability:.6f}"
)

P(NOUN | DET) = 0.586959


In [17]:
emission_counts = defaultdict(Counter)

for sentence in train_sentences:

    for word, tag in sentence:

        word = word.lower()

        emission_counts[tag][word] += 1

print("Emission counts created successfully.")

Emission counts created successfully.


In [18]:
print("Most common NOUN words:")

for word, count in emission_counts["NOUN"].most_common(10):
    print(f"{word}: {count}")

Most common NOUN words:
time: 395
people: 239
thanks: 190
way: 188
place: 186
service: 186
number: 177
year: 176
food: 175
day: 168


In [19]:
emission_probabilities = {}

vocabulary = set()

for tag in emission_counts:

    for word in emission_counts[tag]:
        vocabulary.add(word)

vocabulary = sorted(vocabulary)

vocab_size = len(vocabulary)

for tag in tag_set:

    total = sum(
        emission_counts[tag].values()
    )

    emission_probabilities[tag] = {}

    for word in emission_counts[tag]:

        count = emission_counts[tag][word]

        probability = (
            (count + 1) /
            (total + vocab_size)
        )

        emission_probabilities[tag][word] = probability

print("Emission probabilities calculated.")
print("Vocabulary size:", vocab_size)

Emission probabilities calculated.
Vocabulary size: 17113


In [20]:
unknown_emission_probability = {}

for tag in tag_set:

    total = sum(
        emission_counts[tag].values()
    )

    unknown_emission_probability[tag] = (
        1 / (total + vocab_size)
    )

print("Unknown-word probabilities created.")

Unknown-word probabilities created.


In [21]:
word = "the"
tag = "DET"

probability = emission_probabilities[tag].get(
    word,
    unknown_emission_probability[tag]
)

print(
    f"P({word} | {tag}) = {probability:.6f}"
)

P(the | DET) = 0.271310


In [22]:
import math

def viterbi(words):
    
    words = [word.lower() for word in words]
    
    # Viterbi table
    viterbi = []
    backpointer = []
    
    # Initialization
    first_word = words[0]
    
    first_scores = {}
    first_backpointer = {}
    
    for tag in tag_set:
        
        transition_prob = transition_probabilities[
            "<START>"
        ][tag]
        
        emission_prob = emission_probabilities[tag].get(
            first_word,
            unknown_emission_probability[tag]
        )
        
        first_scores[tag] = (
            math.log(transition_prob)
            + math.log(emission_prob)
        )
        
        first_backpointer[tag] = None
    
    viterbi.append(first_scores)
    backpointer.append(first_backpointer)
    
    # Recursion
    for word in words[1:]:
        
        current_scores = {}
        current_backpointer = {}
        
        for current_tag in tag_set:
            
            emission_prob = emission_probabilities[
                current_tag
            ].get(
                word,
                unknown_emission_probability[current_tag]
            )
            
            best_score = float("-inf")
            best_previous_tag = None
            
            for previous_tag in tag_set:
                
                transition_prob = transition_probabilities[
                    previous_tag
                ].get(
                    current_tag,
                    1e-10
                )
                
                score = (
                    viterbi[-1][previous_tag]
                    + math.log(transition_prob)
                    + math.log(emission_prob)
                )
                
                if score > best_score:
                    best_score = score
                    best_previous_tag = previous_tag
            
            current_scores[current_tag] = best_score
            current_backpointer[current_tag] = best_previous_tag
        
        viterbi.append(current_scores)
        backpointer.append(current_backpointer)
    
    # Termination
    best_final_tag = None
    best_final_score = float("-inf")
    
    for tag in tag_set:
        
        end_probability = transition_probabilities[
            tag
        ].get("<END>", 1e-10)
        
        score = (
            viterbi[-1][tag]
            + math.log(end_probability)
        )
        
        if score > best_final_score:
            best_final_score = score
            best_final_tag = tag
    
    # Backtracking
    best_tags = [best_final_tag]
    
    for i in range(
        len(words) - 1,
        0,
        -1
    ):
        
        previous_tag = backpointer[i][
            best_tags[-1]
        ]
        
        best_tags.append(previous_tag)
    
    best_tags.reverse()
    
    return best_tags

In [23]:
sentence = "The student reads a book"

words = sentence.split()

predicted_tags = viterbi(words)

print("Sentence:")
print(sentence)

print("\nPredicted POS tags:")
print()

for word, tag in zip(words, predicted_tags):
    print(f"{word:15} → {tag}")

Sentence:
The student reads a book

Predicted POS tags:

The             → DET
student         → NOUN
reads           → ADP
a               → DET
book            → NOUN


In [25]:
sentence = input("Enter a sentence: ")

words = sentence.split()

predicted_tags = viterbi(words)

print("\nPOS Tags:")
print("-" * 30)

for word, tag in zip(words, predicted_tags):
    print(f"{word:15} → {tag}")

Enter a sentence:  The student reads a book



POS Tags:
------------------------------
The             → DET
student         → NOUN
reads           → ADP
a               → DET
book            → NOUN


In [26]:
result_df = pd.DataFrame({
    "Word": words,
    "Predicted POS": predicted_tags
})

result_df

,Word,Predicted POS
0,The,DET
1,student,NOUN
2,reads,ADP
3,a,DET
4,book,NOUN


In [27]:
test_sentence = "The cat is eating food"

words = test_sentence.split()

predicted_tags = viterbi(words)

result_df = pd.DataFrame({
    "Word": words,
    "Predicted POS": predicted_tags
})

result_df

,Word,Predicted POS
0,The,DET
1,cat,NOUN
2,is,AUX
3,eating,VERB
4,food,NOUN


In [28]:
test_sentences = []

for sentence in test_data:
    
    words = []
    tags = []
    
    for token in sentence:
        
        words.append(token["form"])
        tags.append(token["upos"])
    
    test_sentences.append(
        (words, tags)
    )

print(
    "Test sentences:",
    len(test_sentences)
)

Test sentences: 2077


In [29]:
all_predictions = []
all_actual_tags = []

for words, actual_tags in test_sentences:
    
    predicted_tags = viterbi(words)
    
    all_predictions.extend(predicted_tags)
    all_actual_tags.extend(actual_tags)

print(
    "Total predicted tags:",
    len(all_predictions)
)

print(
    "Total actual tags:",
    len(all_actual_tags)
)

Total predicted tags: 25450
Total actual tags: 25450


In [30]:
correct = sum(
    predicted == actual
    for predicted, actual
    in zip(
        all_predictions,
        all_actual_tags
    )
)

total = len(all_actual_tags)

accuracy = (
    correct / total
) * 100

print(
    f"Correct predictions: {correct}"
)

print(
    f"Total predictions: {total}"
)

print(
    f"POS Tagging Accuracy: {accuracy:.2f}%"
)

Correct predictions: 21732
Total predictions: 25450
POS Tagging Accuracy: 85.39%


In [31]:
evaluation_report = pd.DataFrame({
    "Metric": [
        "Total Test Tokens",
        "Correct Predictions",
        "Incorrect Predictions",
        "Accuracy (%)"
    ],
    
    "Value": [
        total,
        correct,
        total - correct,
        round(accuracy, 2)
    ]
})

evaluation_report

,Metric,Value
0,Total Test Tokens,25450.00
1,Correct Predictions,21732.00
2,Incorrect Predictions,3718.00
3,Accuracy (%),85.39


In [32]:
sentence = "The student reads a book"

words = sentence.split()

predicted_tags = viterbi(words)

print("=" * 45)
print("           HMM POS TAGGER")
print("=" * 45)

for word, tag in zip(words, predicted_tags):
    print(f"{word:15} → {tag}")

print("=" * 45)
print(f"Test Dataset Accuracy: {accuracy:.2f}%")

           HMM POS TAGGER
The             → DET
student         → NOUN
reads           → ADP
a               → DET
book            → NOUN
Test Dataset Accuracy: 85.39%
